# 02 - Tarea guiada: métricas y evaluación por archivo

En esta tarea **no se usa `course_utils.metrics`**. Primero defines tus propias funciones de métricas y luego las usas para construir una evaluación completa por archivo y por modelo.

Puedes usar `course_utils.data`, `course_utils.palette` y `course_utils.plotting`, pero las métricas son tu responsabilidad.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from course_utils.data import (
    SAMPLE_FILES,
    LEAD_MINUTES,
    get_paths,
    load_sample,
    split_sequence,
    load_prediction,
    make_persistence_prediction,
)
from course_utils.palette import apply_course_style, RAIN_LEVELS
from course_utils.plotting import plot_event_grid, plot_target_prediction_panel, plot_rmse_bias, plot_csi_by_threshold

apply_course_style()
paths = get_paths(PROJECT_ROOT)

THRESHOLDS = [0.5, 2.0, 5.0, 10.0]
THRESHOLD_LABELS = {
    0.5: "lluvia ligera",
    2.0: "lluvia moderada",
    5.0: "lluvia fuerte",
    10.0: "lluvia intensa",
}

print(f"Proyecto: {paths.root}")
print(f"Muestras: {paths.sample_dir}")

## 1. Cargar un caso

Elige una muestra. Si existen predicciones de CasCast/EarthFormer se cargarán; si no, se usará persistencia.

In [ ]:
sample_name = SAMPLE_FILES[0]  # TODO: prueba otros casos
sequence = load_sample(sample_name, paths)
inputs, target = split_sequence(sequence)
prediction, prediction_source = load_prediction(sample_name, inputs, paths)

print(sample_name)
print("Fuente:", prediction_source)
print("inputs:", inputs.shape, "target:", target.shape, "prediction:", prediction.shape)
print("Niveles de color mm/h:", RAIN_LEVELS)

## 2. Visualización inicial

Responde: ¿hacia dónde se mueve la tormenta?, ¿la predicción conserva los máximos?, ¿hay desplazamiento espacial?

In [ ]:
fig = plot_event_grid(inputs, target, prediction, sample_name, prediction_source)
plt.show()

## 3. Definir métricas continuas

Fórmulas:

$$
\mathrm{MAE}_k = \frac{1}{HW}\sum_{i,j}|\hat{Y}_{k,i,j} - Y_{k,i,j}|
$$

$$
\mathrm{RMSE}_k = \sqrt{\frac{1}{HW}\sum_{i,j}(\hat{Y}_{k,i,j} - Y_{k,i,j})^2}
$$

$$
\mathrm{Bias}_k = \frac{1}{HW}\sum_{i,j}(\hat{Y}_{k,i,j} - Y_{k,i,j})
$$

In [ ]:
def safe_pearson_student(x, y):
    """TODO: correlación de Pearson segura para dos campos 2D."""
    # Pistas:
    # 1. Convierte x e y a vectores con .ravel()
    # 2. Elimina valores no finitos con np.isfinite
    # 3. Si alguno tiene desviación estándar cero, devuelve np.nan
    # 4. Usa np.corrcoef(x, y)[0, 1]
    raise NotImplementedError("Completa safe_pearson_student")


def compute_continuous_metrics(pred, target):
    """TODO: devuelve DataFrame con lead_min, MAE, RMSE, Bias, Pearson_r."""
    rows = []
    for i, lead in enumerate(LEAD_MINUTES):
        # TODO: error = pred[i] - target[i]
        # TODO: calcular MAE, RMSE, Bias, Pearson_r
        # TODO: append de un diccionario a rows
        raise NotImplementedError("Completa compute_continuous_metrics")
    return pd.DataFrame(rows)

In [ ]:
# Cuando completes la función, descomenta:
# continuous_student = compute_continuous_metrics(prediction, target)
# continuous_student

In [ ]:
# Cuando tengas continuous_student:
# fig = plot_rmse_bias(continuous_student)
# plt.show()

## 4. Definir métricas binarias

Para umbral $\tau$:

$$
CSI = \frac{TP}{TP + FP + FN}
$$

$$
POD = \frac{TP}{TP + FN}, \qquad FAR = \frac{FP}{TP + FP}
$$

$$
Precision = \frac{TP}{TP + FP}, \qquad Recall = \frac{TP}{TP + FN}
$$

$$
F1 = \frac{2TP}{2TP + FP + FN}
$$

Si el denominador es cero, devuelve `np.nan`.

In [ ]:
def safe_divide(numerator, denominator):
    """TODO: si denominator == 0 devuelve np.nan; si no, numerator / denominator."""
    raise NotImplementedError("Completa safe_divide")


def compute_binary_metrics(pred, target, threshold):
    """TODO: calcula TP, FP, FN, TN, CSI, POD, FAR, Precision, Recall y F1."""
    # TODO: pred_event = pred >= threshold
    # TODO: target_event = target >= threshold
    # TODO: TP, FP, FN, TN con np.logical_and
    # TODO: métricas con safe_divide
    raise NotImplementedError("Completa compute_binary_metrics")


def event_metrics_by_threshold_and_lead_student(pred, target, thresholds=THRESHOLDS):
    """TODO: aplica compute_binary_metrics para cada lead time y umbral."""
    rows = []
    for threshold in thresholds:
        for i, lead in enumerate(LEAD_MINUTES):
            # TODO: metrics = compute_binary_metrics(...)
            # TODO: rows.append({"threshold_mm_h": threshold, "lead_min": int(lead), **metrics})
            raise NotImplementedError("Completa event_metrics_by_threshold_and_lead_student")
    return pd.DataFrame(rows)

In [ ]:
# Cuando completes las funciones, descomenta:
# event_student = event_metrics_by_threshold_and_lead_student(prediction, target, THRESHOLDS)
# event_student.head()

In [ ]:
# Cuando tengas event_student:
# fig = plot_csi_by_threshold(event_student)
# plt.show()

## 5. Pipeline de evaluación por archivo

Ahora integra tus funciones: para cada archivo y cada predicción disponible, calcula promedios por caso.

Esta es la parte más importante de la tarea porque convierte funciones sueltas en un pequeño pipeline reproducible.

In [ ]:
def available_predictions_for_case_student(name):
    """Devuelve target y predicciones disponibles para un archivo."""
    seq = load_sample(name, paths)
    x, y = split_sequence(seq)
    predictions = {"persistence": make_persistence_prediction(x)}

    # Compatibilidad con predicciones opcionales de modelos
    for label, folder in [("earthformer", paths.earthformer_dir), ("cascast", paths.cascast_dir), ("model", paths.model_dir)]:
        pred_path = folder / name
        if pred_path.exists():
            predictions[label] = np.nan_to_num(np.load(pred_path).astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return y, predictions


def evaluate_all_files_student(sample_files=SAMPLE_FILES):
    """TODO: usa tus métricas para construir una tabla por archivo y modelo."""
    rows = []
    for name in sample_files:
        target_case, predictions = available_predictions_for_case_student(name)
        for model_name, pred_case in predictions.items():
            # TODO: continuous_df = compute_continuous_metrics(pred_case, target_case)
            # TODO: event_df = event_metrics_by_threshold_and_lead_student(pred_case, target_case, THRESHOLDS)
            # TODO: agregar RMSE_mean, MAE_mean, Bias_mean, Pearson_mean, CSI_0.5_mean, CSI_5_mean, CSI_10_mean
            raise NotImplementedError("Completa evaluate_all_files_student")
    return pd.DataFrame(rows)

In [ ]:
# Cuando completes el pipeline:
# all_file_metrics = evaluate_all_files_student(SAMPLE_FILES)
# all_file_metrics

In [ ]:
# Resumen global por modelo:
# overall_metrics = (
#     all_file_metrics
#     .groupby("model", as_index=False)
#     [["RMSE_mean", "MAE_mean", "Bias_mean", "Pearson_mean", "CSI_0.5_mean", "CSI_5_mean", "CSI_10_mean"]]
#     .mean(numeric_only=True)
#     .sort_values("RMSE_mean")
# )
# overall_metrics

In [ ]:
# Figura sugerida: fidelidad por caso
# if len(all_file_metrics["model"].unique()) > 1:
#     fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
#     for model_name, group in all_file_metrics.groupby("model"):
#         ax.scatter(group["RMSE_mean"], group["CSI_10_mean"], label=model_name, s=60)
#     ax.set_xlabel("RMSE promedio (mm/h)")
#     ax.set_ylabel("CSI promedio a 10 mm/h")
#     ax.set_title("Error promedio vs lluvia intensa")
#     ax.legend()
#     plt.show()

## 6. Preguntas de análisis

Responde:

1. ¿Por qué CSI suele ser más alto para 0.5 mm/h que para 10 mm/h?
2. ¿Qué modelo o línea base tiene menor RMSE promedio?
3. ¿Qué modelo o línea base tiene mejor CSI a 10 mm/h?
4. ¿Hay casos donde RMSE mejora pero CSI intenso empeora?
5. ¿Qué caso parece más difícil y por qué?

## 7. Reflexión final

Escribe entre 300 y 500 palabras:

> ¿Por qué un modelo puede tener bajo RMSE pero baja utilidad para nowcasting de lluvia intensa?

Tu respuesta debe mencionar suavizado, núcleos intensos, desplazamiento espacial y por qué métricas como CSI ayudan a evaluar eventos relevantes.